In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, row_number, current_timestamp, date_sub, lit
from pyspark.sql.window import Window 
from delta.tables import DeltaTable 
 
# ----------------- Create Spark Session 
spark = SparkSession.builder.appName("Retail_MultiSource_Incremental_ELT").getOrCreate() 
 
# ----------------- Input Paths (Multiple Files) 
 
orders_path = "/Volumes/workspace/default/elt_il_proj_files_v3/Input_Files/orders_*.csv" 
customers_path = "/Volumes/workspace/default/elt_il_proj_files_v3/Input_Files/customers_*.csv" 
payments_path = "/Volumes/workspace/default/elt_il_proj_files_v3/Input_Files/payments_*.json" 
 
bronze_path = "/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/orders_bronze" 
 
# -----------------BRONZE LAYER (ELT - RAW INGESTION)  
orders_df = spark.read.option("header", True).option("inferSchema", True).csv(orders_path) \
   .withColumn("ingestion_time", current_timestamp()) 
 
# Load Bronze (ELT pattern) 
orders_df.write.format("delta").mode("append").save(bronze_path) 
 
# Read Bronze for processing 
bronze_df = spark.read.format("delta").load(bronze_path) 

# -----------------INCREMENTAL + LATE DATA HANDLING (WATERMARK)  
# Default for first-time run
default_date = "1900-01-01"

# Get last processed date (incremental) 
try: 
   last_processed = spark.sql("SELECT max(order_date) as max_date FROM sales_silver ").collect()[0]["max_date"] 
except: 
   last_processed = None  

if last_processed is None:
   last_processed = default_date

# single watermark filter for Late data handling → reprocess last 2 days and incremental as well
bronze_df = bronze_df.filter( 
   col("order_date") >= date_sub(lit(last_processed), 2) 
) 
 
# ----------------TEMP VIEWS CREATION (SQL PROCESSING)  
bronze_df.createOrReplaceTempView("orders_raw") 
 
customers_df = spark.read.option("header", True).csv(customers_path) 
payments_df = spark.read.json(payments_path) 
 
customers_df.createOrReplaceTempView("customers_raw") 
payments_df.createOrReplaceTempView("payments_raw") 

bronze_df.show()
customers_df.show()
payments_df.show()
 
# ------------------DEDUPLICATION (SQL)  
spark.sql(""" 
CREATE OR REPLACE TEMP VIEW orders_dedup AS 
SELECT * 
FROM ( 
   SELECT *, 
          ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY order_date DESC) rn 
   FROM orders_raw 
) t 
WHERE rn = 1 
""") 
 
# ------------------CLEANING + JOIN + DERIVED (SQL)  
spark.sql(""" 
CREATE OR REPLACE TEMP VIEW sales_enriched AS 
SELECT o.*, 
      c.customer_name, 
      p.payment_status, 
      o.amount * 0.18 AS tax_amount, 
      o.amount + (o.amount * 0.18) AS total_amount 
FROM orders_dedup o 
LEFT JOIN customers_raw c ON o.customer_id = c.customer_id 
LEFT JOIN payments_raw p ON o.order_id = p.order_id 
WHERE o.amount > 0 
""") 
 
df_final = spark.sql("SELECT * FROM sales_enriched") 
df_final.show()
 
# ----------------SILVER LAYER (IDEMPOTENT MERGE)  
silver_table = "sales_silver" 
 
if spark.catalog.tableExists(silver_table):
   delta_table = DeltaTable.forName(spark, silver_table) 
 
   delta_table.alias("T").merge(df_final.alias("S"), "T.order_id = S.order_id")\
   .whenMatchedUpdateAll() \
   .whenNotMatchedInsertAll() \
   .execute() 
else: 
   df_final.write.format("delta").saveAsTable(silver_table) 
 
# ------------------GOLD LAYER (SQL AGGREGATION)  
spark.sql(""" 
CREATE OR REPLACE TEMP VIEW sales_final AS 
SELECT * FROM sales_enriched 
""") 

df_gold = spark.sql(""" 
SELECT category, 
      SUM(total_amount) AS total_revenue, 
      COUNT(order_id) AS total_orders 
FROM sales_final 
GROUP BY category 
""") 
 
# -------------------WRITE OUTPUT  
gold_path = "/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/delta/sales_gold" 
parquet_path = "/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/parquet/sales_gold" 

#To remove old files from previous runs - VACUUM -for detla
spark.sql("""
VACUUM delta.`/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/delta/sales_gold` RETAIN 1 HOURS
""")

# To remove folder of parquet
dbutils.fs.rm("/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/parquet/sales_gold", True)

df_gold.write.format("delta").mode("overwrite").save(gold_path) 
df_gold.write.format("parquet").mode("overwrite").save(parquet_path) 
 
print("Production ELT Pipeline Completed Successfully") 
#spark.stop() 

In [0]:
spark.sql("select * from sales_silver").show()

spark.sql("select * from delta.`/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/delta/sales_gold/`").show()

spark.sql("select * from parquet.`/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/parquet/sales_gold/`").show()

In [0]:
spark.sql("select * from sales_silver").show()

spark.sql("select * from delta.`/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/delta/sales_gold/`").show()

spark.sql("select * from parquet.`/Volumes/workspace/default/elt_il_proj_files_v3/Output_Files/parquet/sales_gold/`").show()